# Script 3: Evaluation
**Dilated ResNet + Temporal Attention | 5-Fold Patient-Level Cross-Validation**
Master's Thesis: Deep Learning for Predicting Atrial Fibrillation

1. Run **Script 1** and **Script 2** first — this notebook reads the trained fold models (`model_fold_{k}.keras`), `cv_metadata.json` and the preprocessed cache written by Script 2.
2. Select a GPU runtime (*Runtime → Change runtime type*).
3. Set `DATA_ROOT` in the **USER SETTING** cell to the same SHDB-AF folder used in Scripts 1 and 2.
4. Run all cells top to bottom (*Runtime → Run all*).

All outputs (metrics, plots, reports and confidence intervals) are written to `DATA_ROOT/outputs/results/dilated_resnet_attention/`, next to the models.

In [ ]:
# -*- coding: utf-8 -*-
# =============================================================================
# Script 3: Evaluation
# Dilated ResNet + Temporal Attention  |  5-Fold Patient-Level Cross-Validation
# Master's Thesis: Deep Learning for Predicting Atrial Fibrillation
# =============================================================================
#
# PURPOSE:
#   Evaluates the fold models trained by Script 2. Every saved model is
#   reloaded and re-run on the held-out test patients of its fold. Since each
#   patient is tested in exactly one fold, this yields one out-of-fold
#   prediction per window and per patient. All results are derived from these
#   predictions and grouped in three parts:
#
#   Part A — Performance Evaluation
#            Window- and patient-level metrics (majority-vote and
#            mean-probability aggregation), confusion matrices, ROC and
#            sensitivity/specificity curves, and per-patient reports.
#   Part B — Robustness Diagnostics
#            B1  Cross-fold stability (mean ± SD, training curves)
#            B2  Shortcut check: t-SNE of the penultimate-layer embeddings
#            B3  Time-axis permutation test
#            B4  Intra-patient window correlation and effective sample size
#            B5  Aggregation agreement, patient positive fractions, calibration
#            B6  Integrated Gradients attribution maps and P-wave check
#            B7  Decision Curve Analysis (net benefit)
#   Part C — Confidence Intervals
#            Patient level — Clopper-Pearson, Wilson, DeLong (AUC) and a
#                            percentile bootstrap over patients.
#            Window level  — patient-cluster bootstrap (windows of the same
#                            patient are correlated, see B4).
#            Cross-fold    — Student-t interval over the fold-wise estimates.
#            Decision curve — bootstrap confidence band for the net benefit.
#
#   Diagnostics that point to a potential problem are printed as [FLAG]
#   blocks in the console output.
#
# INPUT (written by Scripts 1 and 2 under DATA_ROOT/outputs):
#   results/<experiment>/cv_metadata.json
#       Fold splits, restored epochs and training histories (Script 2).
#   results/<experiment>/model_fold_{k}.keras
#       Trained model of each fold (Script 2).
#   cache/preprocessed_dataset_all_40.pkl
#       Preprocessed ECG windows (Script 2).
#   preprocessing/train_test_split.json
#       Window index (Script 1); used to look up each patient's recording ID.
#   AdditionalData.csv (in DATA_ROOT)
#       SHDB-AF clinical metadata; used for the patient ages.
#
#   <experiment> = EXPERIMENT_NAME ("dilated_resnet_attention" by default)
#
# OUTPUT FILES (written to results/<experiment>/, next to the models):
#   Part A
#     fold_{k}_*.png, aggregate_*.png            — confusion matrices, ROC and
#                                                  sensitivity/specificity curves
#     Dilated_ResNet_Attention_results.csv       — per-fold patient-level metrics
#     experiment_summary.json                    — experiment description + metrics
#     per_patient_windows_SHDB_AF/patient_*.json — per-patient window accuracy
#     per_patient_window_report.pdf              — per-patient window accuracy
#   Part B
#     B1  crossfold_per_fold_metrics.csv, crossfold_mean_std.csv,
#         crossfold_loss_curves.png, crossfold_accuracy_curves.png
#     B2  tsne_embeddings_class_vs_patient.png, tsne_by_class.png,
#         tsne_by_patient.png
#     B3  permutation_test.csv
#     B4  intra_patient_correlation.csv, intra_patient_correlation_kde.png
#     B5  aggregation_disagreements.csv (only if any), calibration_window.png,
#         calibration_patient.png, patient_fraction_histogram.png
#     B6  integrated_gradients_heatmaps_fold_{k}.png
#     B7  decision_curve_analysis.png
#   Part C
#     confidence_intervals.csv / .json, confidence_intervals_forest_patient.png,
#     decision_curve_analysis_ci.csv, decision_curve_analysis_with_ci.png
#
# DEPENDENCIES:
#   tensorflow, scikit-learn, scipy, numpy, pandas, matplotlib, seaborn
#   (+ standard library). All are preinstalled on Colab.
#
# ENVIRONMENT:
#   Google Colab (a GPU runtime is recommended) or Jupyter.
#   Google Drive is mounted automatically when DATA_ROOT points to it
#   (/content/drive/...); otherwise DATA_ROOT can be any local folder.
#
# USAGE:
#   1. Run Script 1 and Script 2.
#   2. Set DATA_ROOT in the USER SETTING cell to the same folder as in
#      Scripts 1 and 2. All other paths are derived from it.
#   3. Run the script top to bottom.
#
# =============================================================================

## USER SETTING

In [ ]:
# =============================================================================
# USER SETTING — the only line you need to edit
# =============================================================================
# Folder containing the SHDB-AF WFDB records (001.dat / 001.hea / 001.atr, ...)
# and AdditionalData.csv, exactly as downloaded from PhysioNet.
#   Colab example : "/content/drive/MyDrive/shdb-af/1.0.1"
#   Local example : "C:/data/shdb-af/1.0.1"  or  "/home/<user>/data/shdb-af/1.0.1"
#
# Every output of Scripts 1–3 is written under DATA_ROOT/outputs/.

DATA_ROOT = "/path/to/shdb-af/1.0.1"


## SETUP: Drive, Imports

In [ ]:
# Mount Google Drive when DATA_ROOT points to it (Colab only)
if DATA_ROOT.startswith("/content/drive"):
    from google.colab import drive
    drive.mount('/content/drive')

import datetime
import gc
import json
import os
import pickle
import time
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.colors import ListedColormap
from matplotlib.lines import Line2D
import seaborn as sns
from scipy import stats
from scipy.signal import find_peaks

import tensorflow as tf
from tensorflow import keras

from sklearn.calibration import calibration_curve
from sklearn.manifold import TSNE
from sklearn.metrics import (
    accuracy_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, roc_curve, brier_score_loss,
)
from sklearn.neighbors import NearestNeighbors

## CONFIGURATION

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================
# The file paths are derived from DATA_ROOT (USER SETTING cell). The settings
# below reproduce the thesis evaluation.

# ── Paths (derived from DATA_ROOT; identical in Scripts 1–3, do not edit) ─────
CSV_PATH            = os.path.join(DATA_ROOT, "AdditionalData.csv")
OUTPUT_ROOT         = os.path.join(DATA_ROOT, "outputs")
PREPROC_DIR         = os.path.join(OUTPUT_ROOT, "preprocessing")      # Script 1 outputs
SPLIT_FILE          = os.path.join(PREPROC_DIR, "train_test_split.json")
SECONDARY_TEST_FILE = os.path.join(PREPROC_DIR, "secondary_test_set.json")
CACHE_DIR           = os.path.join(OUTPUT_ROOT, "cache")              # Script 2 cache
EXPERIMENT_NAME     = "dilated_resnet_attention"
RESULTS_DIR         = os.path.join(OUTPUT_ROOT, "results", EXPERIMENT_NAME)   # Scripts 2–3

# ── Path checks — stop early with a clear message if a path is wrong ──────────
if not os.path.isdir(DATA_ROOT):
    raise FileNotFoundError(
        f"DATA_ROOT not found: {DATA_ROOT}\n"
        "Set DATA_ROOT in the USER SETTING cell to the SHDB-AF folder."
    )
if not os.path.isfile(os.path.join(RESULTS_DIR, "cv_metadata.json")):
    raise FileNotFoundError(f"cv_metadata.json not found in {RESULTS_DIR}\nRun Script 2 first.")

CONFIG = {

    # ── Input paths (Script 1 and Script 2 outputs) ────────────────────────
    "json_path":            SPLIT_FILE,
    "additional_data_path": CSV_PATH,
    "cache_dir":            CACHE_DIR,
    "cache_filename":       "preprocessed_dataset_all_40.pkl",

    # ── Script 2 results folder ────────────────────────────────────────────
    # The models and cv_metadata.json are read from here, and all evaluation
    # outputs are written here as well.
    "output_dir": RESULTS_DIR,

    # ── Signal and inference (must match Script 2) ─────────────────────────
    "sampling_rate":          200,     # Hz
    "segment_length_samples": 12000,   # 60 s × 200 Hz
    "leads":                  [0],
    "batch_size":             64,
    "random_seed":            42,

    # ── Part B: robustness diagnostic thresholds ───────────────────────────
    "corr_warn_threshold":      0.80,   # B4: mean intra-patient r above this = redundant windows
    "perm_severe_drop_auc":     0.15,   # B3: AUC drop that counts as a collapse to chance
    "perm_chance_auc":          0.60,   # B3: shuffled AUC above this = global-statistics shortcut
    "calib_brier_warn":         0.25,   # B5: Brier score above this = uncalibrated
    "calib_ece_warn":           0.10,   # B5: mean reliability-curve deviation above this = uncalibrated
    "tsne_patient_purity_warn": 0.80,   # B2: kNN patient purity above this = patient memorisation
    "tsne_max_points":          8000,   # B2: subsample cap for t-SNE
    "ig_n_examples":            4,      # B6: top true-positive and false-positive windows per fold
    "ig_steps":                 50,     # B6: Integrated Gradients interpolation steps
    "pwave_attr_frac_warn":     0.15,   # B6: min attribution share in the P-wave / PR region
    "dca_thresholds":           list(np.linspace(0.01, 0.99, 99)),   # B7 / C: threshold grid

    # ── Part C: confidence intervals ───────────────────────────────────────
    "ci_alpha":           0.05,   # 1 - alpha = 95% two-sided intervals
    "ci_n_boot":          2000,   # bootstrap replicates
    "ci_bootstrap_seed":  42,
    "ci_stratified_boot": True,   # resample PAF and non-AF patients separately, so every
                                  # replicate contains both classes (AUC stays defined)
    "ci_width_warn":      0.30,   # flag headline metrics whose CI is wider than this
}

os.makedirs(CONFIG["output_dir"], exist_ok=True)

np.random.seed(CONFIG["random_seed"])
tf.random.set_seed(CONFIG["random_seed"])

## LOADING

In [ ]:
# =============================================================================
# LOADING — Script 2 outputs, console helpers
# =============================================================================

@keras.utils.register_keras_serializable(package="af_attention")
class TemporalAttentionSum(keras.layers.Layer):
    """
    Custom layer of the Script 2 model: sums the attention-weighted features
    over the temporal axis. It must be defined (and registered) before the
    saved models are loaded.
    """
    def call(self, inputs):
        return tf.reduce_sum(inputs, axis=1)

    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[-1])


def banner(msg, ch="="):
    """Print a section header."""
    print("\n" + ch * 70)
    print(msg)
    print(ch * 70)


def flag(msg):
    """Print a robustness warning prominently."""
    line = "!" * 70
    print("\n" + line)
    print("  [FLAG] " + msg)
    print(line)


def load_cv_metadata(config):
    """Load cv_metadata.json (fold splits and training histories) from Script 2."""
    path = os.path.join(config["output_dir"], "cv_metadata.json")
    if not os.path.exists(path):
        raise FileNotFoundError(f"cv_metadata.json not found at {path}. Run Script 2 first.")
    with open(path, "r") as f:
        meta = json.load(f)
    print(f"[LOAD] cv_metadata.json  ({len(meta['folds'])} folds)")
    return meta


def load_dataset(config):
    """Load the preprocessed window cache written by Script 2."""
    cache_path = os.path.join(config["cache_dir"], config["cache_filename"])
    if not os.path.exists(cache_path):
        raise FileNotFoundError(
            f"{config['cache_filename']} not found at {cache_path}. Run Script 2 first."
        )
    with open(cache_path, "rb") as f:
        dataset = pickle.load(f)
    print(f"[LOAD] {config['cache_filename']}  "
          f"({dataset['segments'].shape[0]} windows, "
          f"{len(np.unique(dataset['patient_ids']))} patients)")
    return dataset


def load_fold_model(config, fold_num):
    """Load the trained model of one fold, for inference only (not compiled)."""
    path = os.path.join(config["output_dir"], f"model_fold_{fold_num}.keras")
    if not os.path.exists(path):
        raise FileNotFoundError(f"Model not found: {path}. Run Script 2 first.")
    # safe_mode=False: the models are trusted local files produced by Script 2.
    return keras.models.load_model(path, safe_mode=False, compile=False)


def _load_age_lookup(additional_data_path):
    """Read the patient ages (Subject_ID, Age_at_Holter) from AdditionalData.csv."""
    if not os.path.exists(additional_data_path):
        print(f"  [WARN] AdditionalData.csv not found: {additional_data_path}")
        return None
    df = pd.read_csv(additional_data_path)
    required = {'Subject_ID', 'Age_at_Holter'}
    if required - set(df.columns):
        print(f"  [WARN] AdditionalData.csv missing columns: {required - set(df.columns)}")
        return None
    df = df[['Subject_ID', 'Age_at_Holter']].copy()
    df['Subject_ID']    = df['Subject_ID'].astype(str).str.strip()
    df['Age_at_Holter'] = pd.to_numeric(df['Age_at_Holter'], errors='coerce')
    df = df.dropna(subset=['Age_at_Holter'])
    print(f"  [AGE] Loaded {len(df)} patients from AdditionalData.csv.")
    return df


def _build_patient_age_map(df) -> dict:
    """patient_id -> age at Holter recording."""
    if df is None:
        return {}
    return dict(zip(df['Subject_ID'].astype(str), df['Age_at_Holter'].astype(float)))


def _load_recording_id_lookup(config):
    """patient_id -> recording_id, read from the Script 1 window index (one recording per patient)."""
    path = config.get('json_path', '')
    if not path or not os.path.exists(path):
        print(f"  [WARN] Script 1 window index not found for recording-ID lookup: {path}")
        return {}
    with open(path, 'r') as f:
        data = json.load(f)
    merged = {**data.get('train', {}), **data.get('test', {})}
    lookup = {str(pid): str(entry.get('recording_id', 'N/A')) for pid, entry in merged.items()}
    print(f"  [REC] Loaded {len(lookup)} patient -> recording_id mappings.")
    return lookup

## METRICS & PLOTTING HELPERS

In [ ]:
# =============================================================================
# METRICS & PLOTTING HELPERS
# =============================================================================

def _compute_binary_metrics(true, pred, prob=None) -> dict:
    """Accuracy, sensitivity, specificity, F1, AUC-ROC and confusion matrix."""
    m = {
        'accuracy':         float(accuracy_score(true, pred)),
        'sensitivity':      float(recall_score(true, pred, pos_label=1, zero_division=0)),
        'specificity':      float(recall_score(true, pred, pos_label=0, zero_division=0)),
        'f1':               float(f1_score(true, pred, pos_label=1, zero_division=0)),
        'confusion_matrix': confusion_matrix(true, pred, labels=[0, 1]),
    }
    if prob is not None:
        try:
            m['auc_roc'] = float(roc_auc_score(true, prob))
        except ValueError:
            m['auc_roc'] = float('nan')
    else:
        m['auc_roc'] = float('nan')
    return m


def aggregate_patient_predictions(window_probs, window_preds, patient_ids,
                                  true_labels, tie_breaker=1):
    """
    Aggregate window predictions into one prediction per patient, in two ways:

    Majority vote    — the class predicted for most windows (ties -> tie_breaker).
    Mean probability — the patient's mean window probability, thresholded at 0.5.

    Returns (majority_dict, mean_prob_dict, patient_order), where both dicts
    map patient_id -> {true, pred, mean_prob, n_windows}.
    """
    pat_preds = defaultdict(list)
    pat_probs = defaultdict(list)
    pat_true  = {}
    for prob, pred, pat, true in zip(window_probs, window_preds, patient_ids, true_labels):
        pat_preds[pat].append(int(pred))
        pat_probs[pat].append(float(prob))
        pat_true[pat] = int(true)

    patient_order  = sorted(pat_true.keys())
    majority_dict  = {}
    mean_prob_dict = {}
    for pat in patient_order:
        preds      = pat_preds[pat]
        probs      = pat_probs[pat]
        true_lbl   = pat_true[pat]
        n_wins     = len(preds)
        votes_paf  = sum(preds)
        votes_ctrl = n_wins - votes_paf
        if   votes_paf  > votes_ctrl: majority_pred = 1
        elif votes_ctrl > votes_paf:  majority_pred = 0
        else:                         majority_pred = tie_breaker
        mean_prob      = float(np.mean(probs))
        mean_prob_pred = int(mean_prob >= 0.5)
        majority_dict[pat]  = {'true': true_lbl, 'pred': majority_pred,
                               'mean_prob': mean_prob, 'n_windows': n_wins}
        mean_prob_dict[pat] = {'true': true_lbl, 'pred': mean_prob_pred,
                               'mean_prob': mean_prob, 'n_windows': n_wins}
    return majority_dict, mean_prob_dict, patient_order


def plot_pretty_confusion_matrix(cm, title, output_dir, filename):
    """Save a colour-coded 2×2 confusion matrix (TN, FP / FN, TP)."""
    plt.figure(figsize=(6, 5))
    custom_colors = ['#6ab187', '#fcd76b', '#f99896', '#69b0f3']
    cmap_custom   = ListedColormap(custom_colors)
    color_mask    = np.array([[0, 1], [2, 3]])
    ax = sns.heatmap(
        color_mask, annot=cm, fmt='d', cmap=cmap_custom, cbar=False,
        linewidths=1, linecolor='white',
        annot_kws={"size": 16, "weight": "bold", "color": "black"},
    )
    ax.set_xticklabels(['non-AF (0)', 'PAF (1)'], fontsize=12)
    ax.set_yticklabels(['non-AF (0)', 'PAF (1)'], fontsize=12, rotation=0)
    plt.title(title, fontsize=14, pad=15, fontweight='bold')
    plt.ylabel('True Label (Actual)', fontsize=12, fontweight='bold')
    plt.xlabel('Predicted Label',     fontsize=12, fontweight='bold')
    plt.tight_layout()
    save_path = os.path.join(output_dir, filename)
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"  [CM] Saved -> {save_path}")


def plot_sens_spec_curve(true_y, probs, title, filename, output_dir):
    """Sensitivity and specificity vs. threshold, marking the point where they cross."""
    thresholds = np.linspace(0, 1, 100)
    sensitivities, specificities = [], []
    for t in thresholds:
        preds = (probs >= t).astype(int)
        sensitivities.append(recall_score(true_y, preds, pos_label=1, zero_division=0))
        specificities.append(recall_score(true_y, preds, pos_label=0, zero_division=0))
    plt.figure(figsize=(7, 5))
    plt.plot(thresholds, sensitivities, label='Sensitivity (TPR)', color='blue', linewidth=2)
    plt.plot(thresholds, specificities, label='Specificity (TNR)', color='red',  linewidth=2)
    idx = np.argwhere(
        np.diff(np.sign(np.array(sensitivities) - np.array(specificities)))).flatten()
    if len(idx) > 0:
        opt_t = thresholds[idx[0]]
        plt.plot(opt_t, sensitivities[idx[0]], 'ko', markersize=8,
                 label=f'Intersection (~{opt_t:.2f})')
    plt.axvline(0.5, color='gray', linestyle='--', label='Default Threshold (0.5)')
    plt.title(title, fontsize=14, fontweight='bold')
    plt.xlabel('Probability Threshold', fontsize=12)
    plt.ylabel('Score', fontsize=12)
    plt.legend(loc='lower center')
    plt.grid(alpha=0.3)
    save_path = os.path.join(output_dir, filename)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.show()
    plt.close()
    print(f"  [SAVED] {save_path}")


def plot_sens_spec_curve_youden(true_arr, prob_arr, title, filename, output_dir):
    """Sensitivity and specificity vs. threshold, marking the Youden's J optimum."""
    thresholds = np.linspace(0, 1, 200)
    sensitivities, specificities = [], []
    for thresh in thresholds:
        preds = (prob_arr >= thresh).astype(int)
        sensitivities.append(recall_score(true_arr, preds, pos_label=1, zero_division=0))
        specificities.append(recall_score(true_arr, preds, pos_label=0, zero_division=0))
    j_scores    = np.array(sensitivities) + np.array(specificities) - 1
    best_idx    = np.argmax(j_scores)
    best_thresh = thresholds[best_idx]
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(thresholds, sensitivities, 'r-', linewidth=2, label='Sensitivity (TPR)')
    ax.plot(thresholds, specificities, 'b-', linewidth=2, label='Specificity (TNR)')
    ax.axvline(x=0.5,         color='gray',  linestyle='--', alpha=0.5, label='Threshold = 0.5')
    ax.axvline(x=best_thresh, color='green', linestyle=':',  alpha=0.7,
               label=f"Youden's J optimal = {best_thresh:.3f}")
    ax.set_xlabel('Classification Threshold', fontsize=12)
    ax.set_ylabel('Rate', fontsize=12)
    ax.set_title(title, fontsize=14)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    save_path = os.path.join(output_dir, filename)
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"  [SAVED] {save_path}")
    return best_thresh


def _print_metrics(title, m, n_label=''):
    """Print a metrics box (scalar metrics + confusion matrix)."""
    sfx    = f" ({n_label})" if n_label else ""
    header = f" {title}{sfx} "
    print(f"\n  +{header.center(50, '-')}+")
    for display_name, key in [
        ('Accuracy', 'accuracy'), ('Sensitivity', 'sensitivity'),
        ('Specificity', 'specificity'), ('F1 Score', 'f1'), ('AUC-ROC', 'auc_roc'),
    ]:
        val = m[key]
        val_str = f"{val:.4f}" if isinstance(val, float) else str(val)
        print("  |" + f"  {display_name:<15} : {val_str}".ljust(50) + "|")
    cm = m['confusion_matrix']
    print("  +" + "-" * 50 + "+")
    print("  |" + "  Confusion Matrix [row=True, col=Pred]:".ljust(50) + "|")
    print("  |" + f"  [True] 0   : {cm[0][0]:<13} : {cm[0][1]}".ljust(50) + "|")
    print("  |" + f"  [True] 1   : {cm[1][0]:<13} : {cm[1][1]}".ljust(50) + "|")
    print("  +" + "-" * 50 + "+")


def _print_final(title, m, n_label='', output_dir=None, filename_prefix=None):
    """Print a final metrics box and save its confusion matrix."""
    _print_metrics("FINAL: " + title, m, n_label)
    if output_dir and filename_prefix:
        plot_pretty_confusion_matrix(
            m['confusion_matrix'], title=f"Confusion Matrix - {title}",
            output_dir=output_dir, filename=f"{filename_prefix}_cm.png")


def print_metric_explanations():
    print("\n  Metrics: Sensitivity=TP/(TP+FN); Specificity=TN/(TN+FP); "
          "F1=harmonic mean(Prec,Sens); AUC-ROC=discrimination across thresholds.\n")


LEVEL_KEYS = [('window',           'window_metrics'),
              ('patient_meanprob', 'mean_prob_metrics'),
              ('patient_majority', 'majority_metrics')]


def build_per_fold_table(fold_results):
    """One row per (fold, level) with the metrics of that fold alone (folds are not pooled)."""
    rows = []
    for fr in fold_results:
        for level, key in LEVEL_KEYS:
            m = fr[key]
            rows.append({
                'fold': fr['fold'], 'level': level,
                'accuracy': m['accuracy'], 'sensitivity': m['sensitivity'],
                'specificity': m['specificity'], 'f1': m['f1'], 'auc_roc': m['auc_roc'],
            })
    return pd.DataFrame(rows)

## INFERENCE

In [ ]:
# =============================================================================
# INFERENCE — rebuild all predictions from the saved models
# =============================================================================

def run_inference_all_folds(meta, dataset, config):
    """
    Reload each fold's model and predict on that fold's test windows.

    Returns one dict per fold with the window-level predictions, the
    patient-level aggregations, and the metrics of the fold computed on its
    own test patients.
    """
    banner("RE-RUNNING INFERENCE PER FOLD (from saved .keras models)")
    fold_results = []
    for fmeta in meta["folds"]:
        fold_num = fmeta["fold_number"]
        test_idx = np.array(fmeta["test_indices"])
        test_X   = dataset["segments"][test_idx]
        test_y   = dataset["labels"][test_idx].astype(int)
        pat_ids  = dataset["patient_ids"][test_idx]

        model = load_fold_model(config, fold_num)
        ds_test = tf.data.Dataset.from_tensor_slices(
            test_X.astype(np.float32)).batch(config["batch_size"])
        probs = model.predict(ds_test, verbose=0).flatten()
        preds = (probs >= 0.5).astype(int)
        print(f"  Fold {fold_num}: {len(test_y)} windows, "
              f"{len(np.unique(pat_ids))} patients inferred.")

        win_m = _compute_binary_metrics(test_y, preds, probs)

        maj_dict, mean_dict, pat_order = aggregate_patient_predictions(
            probs, preds, pat_ids.tolist(), test_y)
        true_arr   = np.array([maj_dict[p]['true']       for p in pat_order])
        maj_pred   = np.array([maj_dict[p]['pred']       for p in pat_order])
        mean_prob  = np.array([mean_dict[p]['mean_prob'] for p in pat_order])
        mean_pred  = np.array([mean_dict[p]['pred']      for p in pat_order])
        maj_m  = _compute_binary_metrics(true_arr, maj_pred,  mean_prob)
        mean_m = _compute_binary_metrics(true_arr, mean_pred, mean_prob)

        fold_results.append({
            'fold':              fold_num,
            'test_indices':      test_idx,
            'test_X':            test_X,          # reused by Part B (B2, B3, B6)
            'window_probs':      probs,
            'window_preds':      preds,
            'window_labels':     test_y,
            'window_pat_ids':    pat_ids,
            'pat_order':         pat_order,
            'pat_true':          true_arr,
            'pat_mean_prob':     mean_prob,
            'pat_mean_pred':     mean_pred,
            'pat_maj_pred':      maj_pred,
            'window_metrics':    win_m,
            'majority_metrics':  maj_m,
            'mean_prob_metrics': mean_m,
            'meta':              fmeta,
        })
        del model
        keras.backend.clear_session()
        gc.collect()
    return fold_results

## PART A: PERFORMANCE EVALUATION

In [ ]:
# =============================================================================
# PART A — PERFORMANCE EVALUATION
# =============================================================================
# Per-fold results, plus aggregate results obtained by pooling the test
# windows of all folds. Every patient is tested in exactly one fold, so the
# pooled set holds one out-of-fold prediction for each of the 40 patients.

def _count_patient_windows(pooled):
    """patient_id -> {label, correct, incorrect, total} over the pooled windows."""
    counts = defaultdict(lambda: {'label': None, 'correct': 0, 'incorrect': 0, 'total': 0})
    for pred, label, pat in zip(pooled['preds'], pooled['labels'], pooled['pat_ids']):
        c = counts[str(pat)]
        c['label'] = int(label)
        c['total'] += 1
        if int(pred) == int(label):
            c['correct'] += 1
        else:
            c['incorrect'] += 1
    return counts


def save_per_patient_window_analysis(pooled, config, dataset_name='SHDB_AF'):
    """Print, and save as one JSON per patient, the share of correctly classified windows."""
    banner(f"PER-PATIENT WINDOW ANALYSIS  —  {dataset_name.upper()}")
    age_map    = _build_patient_age_map(_load_age_lookup(config['additional_data_path']))
    rec_lookup = _load_recording_id_lookup(config)
    counts     = _count_patient_windows(pooled)

    results_dict = {}
    for pid in sorted(counts.keys()):
        c     = counts[pid]
        total = c['total']
        correct_pct   = 100.0 * c['correct']   / total
        incorrect_pct = 100.0 * c['incorrect'] / total
        age_str       = str(int(age_map[pid])) if pid in age_map else "N/A"
        results_dict[pid] = {
            'patient_id': pid, 'recording_id': rec_lookup.get(pid, 'N/A'),
            'age': float(age_map[pid]) if pid in age_map else None,
            'total_windows': total,
            'correct_count': c['correct'], 'incorrect_count': c['incorrect'],
            'correct_pct': round(correct_pct, 1), 'incorrect_pct': round(incorrect_pct, 1),
        }
        print(f"Patient {pid} (Age: {age_str}) : Total windows: {total} : "
              f"{c['correct']} correct ({correct_pct:.1f}%), "
              f"{c['incorrect']} incorrect ({incorrect_pct:.1f}%)")

    patient_json_dir = os.path.join(config['output_dir'], f'per_patient_windows_{dataset_name}')
    os.makedirs(patient_json_dir, exist_ok=True)
    for pid, pdata in results_dict.items():
        with open(os.path.join(patient_json_dir, f'patient_{pid}.json'), 'w') as f:
            json.dump(pdata, f, indent=2)
    print(f"\n  [SAVED] {len(results_dict)} per-patient JSON files -> {patient_json_dir}")
    return results_dict


def _generate_summary_pdf(summary_lines, output_path, dataset_name='dataset'):
    """Write a title page followed by the summary lines (45 per page) to a PDF."""
    lines_per_page = 45
    with PdfPages(output_path) as pdf:
        fig, ax = plt.subplots(figsize=(8.5, 11))
        ax.axis('off')
        ax.text(0.5, 0.55,
                f"Per-Patient Window Classification Analysis\n\n"
                f"Dataset  : {dataset_name.upper()}\n"
                f"Generated: {datetime.datetime.now().strftime('%Y-%m-%d  %H:%M')}\n"
                f"Patients : {len(summary_lines)}",
                transform=ax.transAxes, ha='center', va='center', fontsize=14,
                bbox=dict(boxstyle='round,pad=0.8', facecolor='lightsteelblue', alpha=0.5))
        pdf.savefig(fig, bbox_inches='tight')
        plt.show()
        plt.close(fig)
        for page_start in range(0, len(summary_lines), lines_per_page):
            fig, ax = plt.subplots(figsize=(8.5, 11))
            ax.axis('off')
            ax.text(0.02, 0.97, "\n".join(summary_lines[page_start:page_start + lines_per_page]),
                    transform=ax.transAxes, ha='left', va='top',
                    fontsize=8, fontfamily='monospace')
            pdf.savefig(fig, bbox_inches='tight')
            plt.show()
            plt.close(fig)


def generate_pdf_report(pooled, config):
    """Per-patient window accuracy report (per_patient_window_report.pdf)."""
    print("\n[REPORT] Compiling per-patient PDF report...")
    class_names = {0: 'non-AF (0)', 1: 'PAF (1)'}
    counts = _count_patient_windows(pooled)
    summary_lines = []
    for pid in sorted(counts.keys()):
        d             = counts[pid]
        total         = d['total']
        correct_pct   = (d['correct']   / total) * 100 if total > 0 else 0
        incorrect_pct = (d['incorrect'] / total) * 100 if total > 0 else 0
        cls           = class_names.get(d['label'], str(d['label']))
        summary_lines.append(
            f"Patient {pid:<10} | Class: {cls:<10} | Windows: {total:<4} | "
            f"Correct: {d['correct']:<4} ({correct_pct:>5.1f}%) | "
            f"Incorrect: {d['incorrect']:<4} ({incorrect_pct:>5.1f}%)")
    pdf_path = os.path.join(config['output_dir'], 'per_patient_window_report.pdf')
    _generate_summary_pdf(summary_lines, pdf_path, dataset_name='SHDB_AF')
    print(f"  [SAVED] PDF Report -> {pdf_path}")


def evaluate_performance(fold_results, meta, config):
    """
    Part A: per-fold and aggregate metrics, plots, summary files and
    per-patient reports.

    Returns
    -------
    pooled : window-level arrays pooled over all folds (probs, preds, labels, pat_ids)
    agg    : patient-level arrays for the pooled set (true, maj_pred, mean_prob,
             mean_pred, pat_order)
    """
    banner("PART A — PERFORMANCE EVALUATION", "#")

    # ── Per-fold confusion matrices and sensitivity/specificity curves ────
    for fr in fold_results:
        fn = fr['fold']
        plot_pretty_confusion_matrix(
            fr['window_metrics']['confusion_matrix'],
            f"Fold {fn} - Window-Level CM", config['output_dir'],
            f"fold_{fn}_window_cm.png")
        plot_pretty_confusion_matrix(
            fr['majority_metrics']['confusion_matrix'],
            f"Fold {fn} - Patient-Level CM (Majority)", config['output_dir'],
            f"fold_{fn}_patient_cm_majority.png")
        plot_pretty_confusion_matrix(
            fr['mean_prob_metrics']['confusion_matrix'],
            f"Fold {fn} - Patient-Level CM (Mean Prob)", config['output_dir'],
            f"fold_{fn}_patient_cm.png")
        plot_sens_spec_curve(
            fr['pat_true'], fr['pat_mean_prob'],
            f"Fold {fn} - Sensitivity/Specificity Curve",
            f"fold_{fn}_sens_spec.png", config['output_dir'])
        _print_metrics(f"Fold {fn} Window-level", fr['window_metrics'],
                       f"{len(fr['window_labels'])} windows")
        _print_metrics(f"Fold {fn} Patient — Mean Prob", fr['mean_prob_metrics'],
                       f"{len(fr['pat_order'])} patients")

    # ── Patient-level ROC curves of all folds on one figure ───────────────
    print("\n[REPORT] Generating Aggregate ROC Curve...")
    plt.figure(figsize=(7, 6))
    for fr in fold_results:
        fpr, tpr, _ = roc_curve(fr['pat_true'], fr['pat_mean_prob'])
        auc = fr['mean_prob_metrics']['auc_roc']
        plt.plot(fpr, tpr, label=(f"Fold {fr['fold']} (AUC={auc:.3f}, "
                 f"ep={fr['meta']['total_epochs']}, "
                 f"best_val={fr['meta']['best_val_auc']:.3f})"))
    plt.plot([0, 1], [0, 1], 'k--', label='Random (AUC=0.5)')
    plt.title(f'Patient-Level ROC Curve ({len(fold_results)} Folds)')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate (Sensitivity)')
    plt.legend(fontsize=8)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    roc_path = os.path.join(config['output_dir'], "aggregate_roc_curves.png")
    plt.savefig(roc_path, dpi=150)
    plt.show()
    plt.close()
    print(f"  [SAVED] {roc_path}")

    # ── Aggregate: pool the test windows of all folds ─────────────────────
    pooled = {
        'probs':   np.concatenate([fr['window_probs']   for fr in fold_results]),
        'preds':   np.concatenate([fr['window_preds']   for fr in fold_results]),
        'labels':  np.concatenate([fr['window_labels']  for fr in fold_results]),
        'pat_ids': np.concatenate([fr['window_pat_ids'] for fr in fold_results]),
    }
    maj_dict_agg, mean_dict_agg, pat_order_agg = aggregate_patient_predictions(
        pooled['probs'], pooled['preds'], pooled['pat_ids'].tolist(), pooled['labels'])
    true_arr_agg  = np.array([maj_dict_agg[p]['true']       for p in pat_order_agg])
    maj_pred_agg  = np.array([maj_dict_agg[p]['pred']       for p in pat_order_agg])
    mean_prob_agg = np.array([mean_dict_agg[p]['mean_prob'] for p in pat_order_agg])
    mean_pred_agg = np.array([mean_dict_agg[p]['pred']      for p in pat_order_agg])

    plot_sens_spec_curve_youden(
        true_arr_agg, mean_prob_agg,
        f'Patient-Level Sensitivity vs. Specificity (All {len(pat_order_agg)} Patients)',
        'aggregate_sens_spec_curve.png', config['output_dir'])

    agg_window_metrics   = _compute_binary_metrics(pooled['labels'], pooled['preds'], pooled['probs'])
    agg_majority_metrics = _compute_binary_metrics(true_arr_agg, maj_pred_agg, mean_prob_agg)
    agg_prob_metrics     = _compute_binary_metrics(true_arr_agg, mean_pred_agg, mean_prob_agg)

    # ── Per-fold summary table ─────────────────────────────────────────────
    summary_df = pd.DataFrame([{
        'Fold':             fr['fold'],
        'Actual Epochs':    fr['meta']['total_epochs'],
        'Best Val AUC':     fr['meta']['best_val_auc'],
        'Final Train AUC':  fr['meta']['final_train_auc'],
        'AUC Gap':          fr['meta']['final_train_auc'] - fr['meta']['best_val_auc'],
        'AUC (Mean Prob)':  fr['mean_prob_metrics']['auc_roc'],
        'Acc (Mean Prob)':  fr['mean_prob_metrics']['accuracy'],
        'Sens (Mean Prob)': fr['mean_prob_metrics']['sensitivity'],
        'Spec (Mean Prob)': fr['mean_prob_metrics']['specificity'],
        'Acc (Majority)':   fr['majority_metrics']['accuracy'],
        'Sens (Majority)':  fr['majority_metrics']['sensitivity'],
        'Spec (Majority)':  fr['majority_metrics']['specificity'],
    } for fr in fold_results])
    summary_df.loc['Mean'] = summary_df.mean()

    banner(f"FINAL {len(fold_results)}-FOLD CV AGGREGATED METRICS (PATIENT LEVEL)")
    print(summary_df.to_string())

    _print_final("Window-level (aggregate)", agg_window_metrics,
                 f"{len(pooled['labels'])} windows",
                 output_dir=config['output_dir'], filename_prefix="aggregate_window")
    _print_final("Patient-level — Majority Vote (aggregate)", agg_majority_metrics,
                 f"{len(pat_order_agg)} patients",
                 output_dir=config['output_dir'], filename_prefix="aggregate_patient_majority")
    _print_final("Patient-level — Mean Probability (aggregate)", agg_prob_metrics,
                 f"{len(pat_order_agg)} patients",
                 output_dir=config['output_dir'], filename_prefix="aggregate_patient_meanprob")
    print_metric_explanations()

    csv_path = os.path.join(config['output_dir'], "Dilated_ResNet_Attention_results.csv")
    summary_df.to_csv(csv_path)
    print(f"\n  [SAVED] CSV -> {csv_path}")

    # ── experiment_summary.json ────────────────────────────────────────────
    summary_json = {
        "experiment":      "Dilated_ResNet_Attention_CV_Evaluation",
        "experiment_date": meta.get("experiment_date", datetime.datetime.now().isoformat()),
        "config":          meta.get("config", {}),
        "architecture": {
            "core": "Dilated ResNet ([1,2], [2,4], [4,8])",
            "attention": "Temporal Softmax Head",
            "augmentation": "Native TF Amplitude Scaling",
        },
        "training_protocol": {
            "cv_splitter":    "StratifiedGroupKFold",
            "early_stopping": "monitor=val_auc, mode=max, restore_best_weights=True",
        },
        "aggregated_metrics": json.loads(summary_df.to_json()),
    }
    with open(os.path.join(config['output_dir'], "experiment_summary.json"), 'w') as f:
        json.dump(summary_json, f, indent=4)
    print("  [SAVED] experiment_summary.json")

    # ── Per-patient reports ────────────────────────────────────────────────
    save_per_patient_window_analysis(pooled, config, dataset_name='SHDB_AF')
    generate_pdf_report(pooled, config)

    return pooled, {
        'true': true_arr_agg, 'maj_pred': maj_pred_agg,
        'mean_prob': mean_prob_agg, 'mean_pred': mean_pred_agg,
        'pat_order': pat_order_agg,
    }

## PART B1: CROSS-FOLD STABILITY

In [ ]:
# =============================================================================
# PART B1 — CROSS-FOLD STABILITY (mean ± SD; training curves)
# =============================================================================

def analyze_crossfold_stability(fold_results, config):
    """
    Mean ± SD of each metric over the folds (window and patient level), an
    outlier check on the patient-level AUC, and the overlaid train/val loss
    and accuracy curves of all folds.
    """
    banner("B1  CROSS-FOLD STABILITY (Mean +/- SD, window & patient levels)", "#")

    per_fold_df = build_per_fold_table(fold_results)

    metric_cols = ['accuracy', 'sensitivity', 'specificity', 'f1', 'auc_roc']
    summary_rows = []
    for level in per_fold_df['level'].unique():
        sub = per_fold_df[per_fold_df['level'] == level]
        for col in metric_cols:
            summary_rows.append({
                'level': level, 'metric': col,
                'mean': sub[col].mean(), 'std': sub[col].std(ddof=1),
                'min': sub[col].min(), 'max': sub[col].max(),
            })
    summary_stats = pd.DataFrame(summary_rows)

    print("\n[PER-FOLD BREAKDOWN] (identify outlier folds):")
    print(per_fold_df.round(4).to_string(index=False))
    print("\n[CROSS-FOLD Mean +/- SD]:")
    for _, r in summary_stats.iterrows():
        print(f"  {r['level']:<18} {r['metric']:<12} : "
              f"{r['mean']:.4f} +/- {r['std']:.4f}  "
              f"[min {r['min']:.4f}, max {r['max']:.4f}]")

    per_fold_df.to_csv(os.path.join(config['output_dir'], "crossfold_per_fold_metrics.csv"),
                       index=False)
    summary_stats.to_csv(os.path.join(config['output_dir'], "crossfold_mean_std.csv"),
                         index=False)
    print("\n  [SAVED] crossfold_per_fold_metrics.csv, crossfold_mean_std.csv")

    # Outlier check: a fold whose patient-level AUC is more than 2 SD below the mean
    pat_auc = per_fold_df[per_fold_df['level'] == 'patient_meanprob'][['fold', 'auc_roc']]
    mu, sd = pat_auc['auc_roc'].mean(), pat_auc['auc_roc'].std(ddof=1)
    for _, r in pat_auc.iterrows():
        if sd > 0 and r['auc_roc'] < mu - 2 * sd:
            flag(f"Fold {int(r['fold'])} is a patient-AUC outlier "
                 f"({r['auc_roc']:.3f} < mean-2SD={mu - 2*sd:.3f}).")

    # Train/val curves, truncated at the restored-weights epoch (from cv_metadata)
    colors = plt.cm.tab10(np.linspace(0, 1, len(fold_results)))
    for metric, val_metric, ylabel, fname in [
        ('loss', 'val_loss', 'Loss', 'crossfold_loss_curves.png'),
        ('accuracy', 'val_accuracy', 'Accuracy', 'crossfold_accuracy_curves.png'),
    ]:
        plt.figure(figsize=(9, 6))
        for fr, c in zip(fold_results, colors):
            h = fr['meta']['history_truncated']
            tr, va = h[metric], h[val_metric]
            ep = np.arange(1, len(tr) + 1)
            plt.plot(ep, tr, '-',  color=c, linewidth=1.8,
                     label=f"Fold {fr['fold']} train")
            plt.plot(ep, va, '--', color=c, linewidth=1.8,
                     label=f"Fold {fr['fold']} val")
            # Marker at the restored-weights epoch
            plt.plot(ep[-1], tr[-1], 'o', color=c, markersize=8)
            plt.plot(ep[-1], va[-1], 'o', color=c, markersize=8,
                     markerfacecolor='white')
        plt.title(f"Truncated Train/Val {ylabel} per Fold "
                  f"(solid=train, dashed=val, dot=restored epoch)")
        plt.xlabel("Epoch (truncated at restored-weights epoch)")
        plt.ylabel(ylabel)
        plt.legend(fontsize=8, ncol=2)
        plt.grid(alpha=0.3)
        plt.tight_layout()
        p = os.path.join(config['output_dir'], fname)
        plt.savefig(p, dpi=150)
        plt.show()
        plt.close()
        print(f"  [SAVED] {p}")

    return per_fold_df, summary_stats

## PART B2: SHORTCUT CHECK (t-SNE)

In [ ]:
# =============================================================================
# PART B2 — SHORTCUT / MEMORISATION CHECK (penultimate embeddings -> t-SNE)
# =============================================================================

def analyze_embeddings_tsne(fold_results, config):
    """
    Project the 'fc_dense' embeddings of all test windows with t-SNE, coloured
    by class and by patient. If windows cluster by patient rather than by
    class (high kNN patient purity), the model may be memorising
    patient-specific signatures.
    """
    banner("B2  SHORTCUT CHECK (fc_dense embeddings -> t-SNE by class & patient)", "#")

    embeds, classes, patients = [], [], []
    for fr in fold_results:
        model = load_fold_model(config, fr['fold'])
        embed_model = keras.Model(model.input, model.get_layer('fc_dense').output)
        ds = tf.data.Dataset.from_tensor_slices(
            fr['test_X'].astype(np.float32)).batch(config['batch_size'])
        emb = embed_model.predict(ds, verbose=0)
        embeds.append(emb)
        classes.append(fr['window_labels'])
        patients.append(fr['window_pat_ids'])
        del model, embed_model
        keras.backend.clear_session()
        gc.collect()

    embeds   = np.concatenate(embeds, axis=0)
    classes  = np.concatenate(classes, axis=0)
    patients = np.concatenate(patients, axis=0)

    n = embeds.shape[0]
    cap = config['tsne_max_points']
    if n > cap:
        sel = np.random.RandomState(config['random_seed']).choice(n, cap, replace=False)
        embeds_s, classes_s, patients_s = embeds[sel], classes[sel], patients[sel]
        print(f"  [t-SNE] Subsampled {cap}/{n} windows for tractability.")
    else:
        embeds_s, classes_s, patients_s = embeds, classes, patients

    print(f"  [t-SNE] Running on {embeds_s.shape[0]} x {embeds_s.shape[1]} embeddings...")
    coords = TSNE(n_components=2, random_state=config['random_seed'],
                  init='pca', perplexity=30).fit_transform(embeds_s)

    uniq_pat = sorted(set(patients_s.tolist()))
    pat_to_int = {p: i for i, p in enumerate(uniq_pat)}
    pat_int = np.array([pat_to_int[p] for p in patients_s])

    class_colors = np.where(classes_s == 1, '#c44e52', '#4c72b0')   # PAF red, non-AF blue
    pat_cmap = plt.get_cmap('tab20', len(uniq_pat))

    def _draw_class_panel(ax):
        ax.scatter(coords[:, 0], coords[:, 1], c=class_colors, s=6, alpha=0.6)
        ax.set_title("t-SNE of Penultimate Embeddings — Colored by Class")
        handles = [
            Line2D([0], [0], marker='o', linestyle='', color='#c44e52',
                   label='PAF (class 1)'),
            Line2D([0], [0], marker='o', linestyle='', color='#4c72b0',
                   label='non-AF (class 0)'),
        ]
        ax.legend(handles=handles, loc='best', fontsize=9, title='Class')

    def _draw_patient_panel(ax):
        ax.scatter(coords[:, 0], coords[:, 1], c=pat_int, cmap=pat_cmap, s=6, alpha=0.6)
        ax.set_title(f"t-SNE of Penultimate Embeddings — Colored by Patient ID "
                     f"({len(uniq_pat)} patients)")
        handles = [Line2D([0], [0], marker='o', linestyle='', color=pat_cmap(i),
                          label=str(p)) for i, p in enumerate(uniq_pat)]
        ax.legend(handles=handles, loc='center left', bbox_to_anchor=(1.02, 0.5),
                  fontsize=6, ncol=2, title='Patient ID', frameon=True)

    def _save(fig_name):
        p = os.path.join(config['output_dir'], fig_name)
        plt.tight_layout()
        plt.savefig(p, dpi=150, bbox_inches='tight')
        plt.show()
        plt.close()
        print(f"  [SAVED] {p}")

    # Combined figure (class | patient)
    _, axes = plt.subplots(1, 2, figsize=(20, 7))
    _draw_class_panel(axes[0])
    _draw_patient_panel(axes[1])
    for ax in axes:
        ax.set_xlabel("t-SNE 1")
        ax.set_ylabel("t-SNE 2")
    _save("tsne_embeddings_class_vs_patient.png")

    # Class only
    _, ax1 = plt.subplots(figsize=(8, 7))
    _draw_class_panel(ax1)
    ax1.set_xlabel("t-SNE 1")
    ax1.set_ylabel("t-SNE 2")
    _save("tsne_by_class.png")

    # Patient only
    _, ax2 = plt.subplots(figsize=(10, 7))
    _draw_patient_panel(ax2)
    ax2.set_xlabel("t-SNE 1")
    ax2.set_ylabel("t-SNE 2")
    _save("tsne_by_patient.png")

    # kNN purity in the embedding space: share of each window's k nearest
    # neighbours that belong to the same patient / the same class
    k = 10
    nn = NearestNeighbors(n_neighbors=k + 1).fit(embeds_s)
    _, idx = nn.kneighbors(embeds_s)
    idx = idx[:, 1:]  # drop the window itself
    same_patient = np.mean([np.mean(patients_s[nbrs] == patients_s[i])
                            for i, nbrs in enumerate(idx)])
    same_class   = np.mean([np.mean(classes_s[nbrs] == classes_s[i])
                            for i, nbrs in enumerate(idx)])
    print(f"\n  kNN(k={k}) neighbor purity — patient: {same_patient:.3f} | "
          f"class: {same_class:.3f}")
    if same_patient > config['tsne_patient_purity_warn']:
        flag(f"Embeddings cluster by PATIENT (purity {same_patient:.3f} > "
             f"{config['tsne_patient_purity_warn']}). Possible memorization of "
             f"patient-specific signatures -> retrain with stricter regularization / "
             f"patient-stratified splits.")
    else:
        print(f"  [OK] No dominant patient-identity clustering "
              f"(patient purity {same_patient:.3f}).")

## PART B3: TIME-AXIS PERMUTATION TEST

In [ ]:
# =============================================================================
# PART B3 — TIME-AXIS PERMUTATION TEST
# =============================================================================

def _shuffle_time_axis(X, seed, chunk=512):
    """Randomly permute the samples of every window along the time axis (in chunks, to save memory)."""
    rng = np.random.RandomState(seed)
    out = np.empty_like(X)
    T = X.shape[1]
    for s in range(0, X.shape[0], chunk):
        e = min(s + chunk, X.shape[0])
        block = X[s:e]
        perm = np.argsort(rng.rand(block.shape[0], T), axis=1)   # one permutation per window
        out[s:e] = np.take_along_axis(block, perm[:, :, None], axis=1)
    return out


def run_permutation_test(fold_results, config):
    """
    Re-run each frozen model on test windows whose samples were shuffled in
    time. Shuffling destroys the waveform morphology but keeps the amplitude
    distribution, so a model that relies on morphology should drop towards
    chance; if the AUC stays high, the model exploits global signal statistics.
    """
    banner("B3  DATA PERMUTATION (shuffle time axis=1, frozen model)", "#")
    rows = []
    for fr in fold_results:
        model = load_fold_model(config, fr['fold'])
        Xs = _shuffle_time_axis(fr['test_X'].astype(np.float32),
                                seed=config['random_seed'] + fr['fold'])
        ds = tf.data.Dataset.from_tensor_slices(Xs).batch(config['batch_size'])
        probs_s = model.predict(ds, verbose=0).flatten()
        preds_s = (probs_s >= 0.5).astype(int)
        y = fr['window_labels']

        base_auc = fr['window_metrics']['auc_roc']
        base_acc = fr['window_metrics']['accuracy']
        try:
            shuf_auc = float(roc_auc_score(y, probs_s))
        except ValueError:
            shuf_auc = float('nan')
        shuf_acc = float(accuracy_score(y, preds_s))

        d_auc = base_auc - shuf_auc
        d_acc = base_acc - shuf_acc
        rows.append({'fold': fr['fold'], 'base_auc': base_auc, 'shuffled_auc': shuf_auc,
                     'auc_drop': d_auc, 'base_acc': base_acc, 'shuffled_acc': shuf_acc,
                     'acc_drop': d_acc})
        print(f"  Fold {fr['fold']}: AUC {base_auc:.3f} -> {shuf_auc:.3f} "
              f"(drop {d_auc:+.3f}) | Acc {base_acc:.3f} -> {shuf_acc:.3f} "
              f"(drop {d_acc:+.3f})")
        del model, Xs
        keras.backend.clear_session()
        gc.collect()

    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(config['output_dir'], "permutation_test.csv"), index=False)
    print("\n  [SAVED] permutation_test.csv")

    mean_shuf = df['shuffled_auc'].mean()
    mean_drop = df['auc_drop'].mean()
    if mean_shuf > config['perm_chance_auc']:
        flag(f"Shuffled-time AUC stays high (mean {mean_shuf:.3f} > "
             f"{config['perm_chance_auc']}). Model may exploit GLOBAL STATISTICAL "
             f"ARTIFACTS rather than temporal waveform morphology.")
    elif mean_drop >= config['perm_severe_drop_auc']:
        print(f"  [OK] Severe AUC collapse toward chance (mean drop {mean_drop:.3f}) "
              f"-> model relies on physiological waveform shape (expected).")
    else:
        flag(f"Small AUC drop on time-shuffle (mean drop {mean_drop:.3f}). "
             f"Model likely NOT learning temporal morphology.")
    return df

## PART B4: INTRA-PATIENT CORRELATION

In [ ]:
# =============================================================================
# PART B4 — INTRA-PATIENT CORRELATION + EFFECTIVE SAMPLE SIZE
# =============================================================================

def analyze_intra_patient_correlation(dataset, config):
    """
    Mean pairwise correlation between the windows of each patient, and the
    resulting effective sample size n_eff = n / (1 + (n - 1) * r). Highly
    correlated windows mean that the window count greatly overstates the
    amount of independent evidence.
    """
    banner("B4  INTRA-PATIENT PAIRWISE CORRELATION + EFFECTIVE SAMPLE SIZE", "#")
    seg = dataset['segments']
    pids = dataset['patient_ids']
    uniq = sorted(set(pids.tolist()))

    r_scores, n_eff_list, rows = [], [], []
    total_windows = 0
    for p in uniq:
        mask = (pids == p)
        W = seg[mask].reshape(np.sum(mask), -1).astype(np.float32)   # one row per window
        n = W.shape[0]
        total_windows += n
        if n < 2:
            continue
        cmat = np.corrcoef(W)
        iu = np.triu_indices(n, k=1)
        r = float(np.nanmean(cmat[iu]))
        r = max(min(r, 0.999999), -0.999999)
        n_eff = n / (1.0 + (n - 1) * max(r, 0.0))   # equicorrelated-block approximation
        r_scores.append(r)
        n_eff_list.append(n_eff)
        rows.append({'patient': p, 'n_windows': n, 'mean_r': r, 'n_eff': n_eff})

    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(config['output_dir'], "intra_patient_correlation.csv"), index=False)
    r_scores = np.array(r_scores)

    # Patients with constant windows yield NaN correlations; exclude them from
    # the summary and the plot.
    finite_mask = np.isfinite(r_scores)
    n_invalid = int(np.size(r_scores) - np.sum(finite_mask))
    if n_invalid > 0:
        print(f"  [WARN] Dropping {n_invalid} patient(s) with non-finite mean "
              f"intra-patient correlation (degenerate/constant windows).")
    r_scores_valid = r_scores[finite_mask]

    total_eff = float(np.sum(n_eff_list))
    print(f"\n  Patients: {len(r_scores)} | total windows: {total_windows}")
    if r_scores_valid.size > 0:
        print(f"  Mean intra-patient r: {r_scores_valid.mean():.3f} +/- "
              f"{r_scores_valid.std(ddof=1) if r_scores_valid.size > 1 else 0.0:.3f}")
    print(f"  Estimated EFFECTIVE sample size: {total_eff:.1f} windows "
          f"(vs {total_windows} raw windows, {len(uniq)} patients).")
    print("  [SAVED] intra_patient_correlation.csv")

    plt.figure(figsize=(8, 5))
    if r_scores_valid.size == 0:
        print("  [WARN] No valid intra-patient correlation values to plot.")
    else:
        sns.histplot(r_scores_valid, bins=20, kde=(r_scores_valid.size > 1),
                     stat='density', color='#4c72b0')
        # Extend the x-axis beyond [0, 1] when needed so no value is cut off
        lo = min(0.0, float(r_scores_valid.min()) - 0.05)
        hi = max(1.0, float(r_scores_valid.max()) + 0.05)
        plt.xlim(lo, hi)
    plt.axvline(config['corr_warn_threshold'], color='red', linestyle='--',
                label=f"warn threshold r={config['corr_warn_threshold']}")
    plt.xlabel("Mean intra-patient window correlation (r)")
    plt.ylabel("Density")
    plt.title(f"Distribution of Intra-Patient Correlation ({len(uniq)} patients)")
    plt.legend()
    plt.tight_layout()
    p = os.path.join(config['output_dir'], "intra_patient_correlation_kde.png")
    plt.savefig(p, dpi=150)
    plt.show()
    plt.close()
    print(f"  [SAVED] {p}")

    if r_scores_valid.size > 0:
        frac_high = float(np.mean(r_scores_valid > config['corr_warn_threshold']))
        if r_scores_valid.mean() > config['corr_warn_threshold'] or frac_high > 0.5:
            flag(f"High intra-patient redundancy (mean r={r_scores_valid.mean():.3f}, "
                 f"{frac_high*100:.0f}% of patients > {config['corr_warn_threshold']}). "
                 f"i.i.d. assumption violated; true sample size (~{total_eff:.0f}) is much "
                 f"closer to the patient count ({len(uniq)}) than the window count "
                 f"({total_windows}).")
        else:
            print("  [OK] Intra-patient correlation below redundancy threshold.")
    return df

## PART B5: AGGREGATION & CALIBRATION

In [ ]:
# =============================================================================
# PART B5 — AGGREGATION AGREEMENT, POSITIVE FRACTIONS, CALIBRATION
# =============================================================================

def analyze_aggregation_and_calibration(pooled, agg, config):
    """
    (a) Patients for which majority vote and mean probability disagree.
    (b) Histogram of the fraction of PAF-predicted windows per patient.
    (c) Reliability diagrams and Brier score at window and patient level.
    """
    banner("B5  AGGREGATION / FRACTION / CALIBRATION", "#")

    # ── (a) Majority vote vs. mean probability ─────────────────────────────
    probs, preds = pooled['probs'], pooled['preds']
    labels, pat_ids = pooled['labels'], pooled['pat_ids']
    maj_dict, mean_dict, order = aggregate_patient_predictions(
        probs, preds, pat_ids.tolist(), labels)

    pat_win_preds = defaultdict(list)
    for pred, pat in zip(preds, pat_ids):
        pat_win_preds[pat].append(int(pred))

    disagreements = []
    for p in order:
        mj, mp = maj_dict[p]['pred'], mean_dict[p]['pred']
        if mj != mp:
            disagreements.append({
                'patient': p, 'true': maj_dict[p]['true'],
                'majority_pred': mj, 'mean_prob_pred': mp,
                'mean_prob': round(mean_dict[p]['mean_prob'], 4),
                'n_windows': maj_dict[p]['n_windows'],
                'frac_pos_windows': round(float(np.mean(pat_win_preds[p])), 4),
            })
    print("\n  Majority-vote vs Mean-probability aggregation:")
    if disagreements:
        ddf = pd.DataFrame(disagreements)
        ddf.to_csv(os.path.join(config['output_dir'], "aggregation_disagreements.csv"),
                   index=False)
        print(ddf.to_string(index=False))
        flag(f"{len(disagreements)} patient(s) DISAGREE between majority-vote and "
             f"mean-probability aggregation (borderline/volatile): "
             f"{[d['patient'] for d in disagreements]}")
    else:
        print("  [OK] No patients disagree between the two aggregation methods.")

    # ── (b) Fraction of PAF-predicted windows per patient, by true class ───
    fracs_paf, fracs_ctrl = [], []
    for p in order:
        frac = float(np.mean(pat_win_preds[p]))
        if maj_dict[p]['true'] == 1:
            fracs_paf.append(frac)
        else:
            fracs_ctrl.append(frac)
    plt.figure(figsize=(8, 5))
    bins = np.linspace(0, 1, 21)
    plt.hist(fracs_ctrl, bins=bins, alpha=0.6, label='True non-AF', color='#55a868')
    plt.hist(fracs_paf,  bins=bins, alpha=0.6, label='True PAF',    color='#c44e52')
    plt.axvline(0.5, color='gray', linestyle='--', label='0.50 threshold')
    plt.xlim(0, 1)
    plt.xlabel("Fraction of windows predicted PAF")
    plt.ylabel("Patient count")
    plt.title("Per-Patient Positive-Fraction by True Class")
    plt.legend()
    plt.tight_layout()
    p_hist = os.path.join(config['output_dir'], "patient_fraction_histogram.png")
    plt.savefig(p_hist, dpi=150)
    plt.show()
    plt.close()
    print(f"\n  [SAVED] {p_hist}")

    def _share_near_half(fracs):
        """Share of patients whose positive fraction lies in [0.35, 0.65]."""
        return np.mean([(0.35 <= f <= 0.65) for f in fracs]) if fracs else 0.0

    overlap_frac = (_share_near_half(fracs_paf) + _share_near_half(fracs_ctrl)) / 2
    if overlap_frac > 0.3:
        flag(f"Patient fraction scores overlap heavily near 0.50 "
             f"({overlap_frac*100:.0f}% within [0.35,0.65]). Model may rely on a weak, "
             f"coin-flip vote rather than a distinct physiological signal.")
    else:
        print("  [OK] Patient fractions separate away from 0.50.")

    # ── (c) Calibration at window and patient level ────────────────────────
    def _calib(y, prob, level, fname):
        y = np.asarray(y)
        prob = np.asarray(prob)
        brier = brier_score_loss(y, prob)
        frac_pos, mean_pred = calibration_curve(y, prob, n_bins=10, strategy='uniform')
        ece = float(np.mean(np.abs(frac_pos - mean_pred)))
        plt.figure(figsize=(6, 6))
        plt.plot([0, 1], [0, 1], 'k--', label='Perfect (y=x)')
        plt.plot(mean_pred, frac_pos, 'o-', label=f'{level} (Brier={brier:.3f})')
        plt.xlabel("Mean predicted probability")
        plt.ylabel("Observed frequency (PAF)")
        plt.title(f"Reliability Diagram — {level} level")
        plt.legend()
        plt.grid(alpha=0.3)
        plt.tight_layout()
        pp = os.path.join(config['output_dir'], fname)
        plt.savefig(pp, dpi=150)
        plt.show()
        plt.close()
        print(f"  [SAVED] {pp}  (Brier={brier:.4f}, ECE={ece:.4f})")
        return brier, ece

    print("\n  Calibration:")
    bw, ew = _calib(labels, probs, "window", "calibration_window.png")
    bp, ep = _calib(agg['true'], agg['mean_prob'], "patient", "calibration_patient.png")
    for level, b, e in [("window", bw, ew), ("patient", bp, ep)]:
        if b > config['calib_brier_warn'] or e > config['calib_ece_warn']:
            flag(f"{level.capitalize()}-level probabilities are UNCALIBRATED "
                 f"(Brier={b:.3f}, ECE={e:.3f}). Apply Platt scaling / isotonic "
                 f"regression before deployment.")
        else:
            print(f"  [OK] {level}-level calibration acceptable "
                  f"(Brier={b:.3f}, ECE={e:.3f}).")

## PART B6: INTERPRETABILITY (Integrated Gradients)

In [ ]:
# =============================================================================
# PART B6 — INTERPRETABILITY (Integrated Gradients; P-wave check)
# =============================================================================

def _integrated_gradients(model, x, baseline, steps):
    """Integrated Gradients (trapezoidal rule) for a single window x of shape (T, C)."""
    x = x.astype(np.float32)
    alphas = np.linspace(0.0, 1.0, steps + 1).astype(np.float32)
    interp = baseline[None] + alphas[:, None, None] * (x[None] - baseline[None])
    interp = tf.convert_to_tensor(interp)
    with tf.GradientTape() as tape:
        tape.watch(interp)
        preds = model(interp, training=False)
    grads = tape.gradient(preds, interp).numpy()   # (steps+1, T, C)
    avg_grads = (grads[:-1] + grads[1:]) / 2.0
    avg_grads = avg_grads.mean(axis=0)             # (T, C)
    return (x - baseline) * avg_grads              # (T, C)


def _pwave_region_mask(signal_1d, fs):
    """Approximate P-wave / PR region: 200 ms to 50 ms before each detected R-peak."""
    sig = signal_1d - np.median(signal_1d)
    height = np.percentile(np.abs(sig), 98)
    peaks, _ = find_peaks(np.abs(sig), height=height, distance=int(0.3 * fs))
    mask = np.zeros_like(signal_1d, dtype=bool)
    p_start = int(0.20 * fs)   # 200 ms before R
    p_end   = int(0.05 * fs)   # 50 ms before R
    for r in peaks:
        a, b = max(0, r - p_start), max(0, r - p_end)
        mask[a:b] = True
    return mask


def analyze_integrated_gradients(fold_results, config):
    """
    Integrated Gradients heatmaps (zero baseline) for the most confident
    true-positive and false-positive PAF windows of every fold, and the share
    of attribution that falls in the P-wave / PR region.
    """
    banner("B6  INTERPRETABILITY (Integrated Gradients heatmaps; P-wave flag)", "#")

    rec_lookup = _load_recording_id_lookup(config)
    fs = config['sampling_rate']
    baseline = np.zeros((config['segment_length_samples'], len(config['leads'])),
                        dtype=np.float32)
    n_ex = config['ig_n_examples']

    all_pwave_fracs = []
    for fr in fold_results:
        fold_num = fr['fold']
        probs, preds, y = fr['window_probs'], fr['window_preds'], fr['window_labels']

        tp_mask = (preds == 1) & (y == 1)
        fp_mask = (preds == 1) & (y == 0)
        tp_sel = np.where(tp_mask)[0][np.argsort(-probs[tp_mask])][:n_ex]
        fp_sel = np.where(fp_mask)[0][np.argsort(-probs[fp_mask])][:n_ex]
        selected = [('TP', i) for i in tp_sel] + [('FP', i) for i in fp_sel]

        if not selected:
            print(f"  [WARN] Fold {fold_num} has no positive-predicted windows; "
                  f"skipping its Integrated Gradients heatmap.")
            continue

        model = load_fold_model(config, fold_num)
        pwave_fracs = []
        fig, axes = plt.subplots(len(selected), 1, figsize=(12, 3 * len(selected)))
        if len(selected) == 1:
            axes = [axes]
        for ax, (kind, idx) in zip(axes, selected):
            x = fr['test_X'][idx]
            ig = _integrated_gradients(model, x, baseline, config['ig_steps'])
            attr = np.abs(ig[:, 0])
            sig  = x[:, 0]
            t = np.arange(len(sig)) / fs
            ax.plot(t, sig, color='black', linewidth=0.6)
            sc = ax.scatter(t, sig, c=attr, cmap='hot', s=4)

            patient_id   = str(fr['window_pat_ids'][idx])
            recording_id = rec_lookup.get(patient_id, 'N/A')
            window_id    = int(fr['test_indices'][idx])
            ax.set_title(f"{kind}  |  Fold {fold_num}  |  Patient {patient_id}  |  "
                         f"Recording {recording_id}  |  Window {window_id}  |  "
                         f"prob={probs[idx]:.3f}")
            ax.set_xlabel("Time (s)")
            ax.set_ylabel("ECG (z)")
            fig.colorbar(sc, ax=ax, fraction=0.02)

            if attr.sum() > 0:
                mask = _pwave_region_mask(sig, fs)
                pwave_fracs.append(float(attr[mask].sum() / attr.sum()))
        plt.tight_layout()
        p = os.path.join(config['output_dir'], f"integrated_gradients_heatmaps_fold_{fold_num}.png")
        plt.savefig(p, dpi=150)
        plt.show()
        plt.close()
        print(f"  [SAVED] {p}")

        del model
        keras.backend.clear_session()
        gc.collect()

        all_pwave_fracs.extend(pwave_fracs)

    if all_pwave_fracs:
        mean_pfrac = float(np.mean(all_pwave_fracs))
        print(f"\n  Mean attribution share in P/PR region (across all folds): {mean_pfrac:.3f}")
        if mean_pfrac < config['pwave_attr_frac_warn']:
            flag(f"Model largely IGNORES the P-wave / PR-interval region "
                 f"(attribution share {mean_pfrac:.3f} < "
                 f"{config['pwave_attr_frac_warn']}). Possible reliance on "
                 f"non-physiological artifacts rather than atrial markers.")
        else:
            print("  [OK] P-wave region receives meaningful attribution.")
    else:
        print("  [WARN] No attributions available to assess the P-wave region.")

## PART B7: DECISION CURVE ANALYSIS

In [ ]:
# =============================================================================
# PART B7 — DECISION CURVE ANALYSIS (Net Benefit)
# =============================================================================

def run_decision_curve_analysis(agg, config):
    """
    Patient-level net benefit, NB(p_t) = TP/N - FP/N * p_t / (1 - p_t),
    compared with the Treat-All and Treat-None strategies.
    """
    banner("B7  DECISION CURVE ANALYSIS (Net Benefit vs Treat-All / Treat-None)", "#")
    y = np.asarray(agg['true'])
    prob = np.asarray(agg['mean_prob'])
    N = len(y)
    prev = y.mean()
    thresholds = np.array(config['dca_thresholds'])

    nb_model, nb_all = [], []
    for pt in thresholds:
        w = pt / (1 - pt)
        pred = (prob >= pt).astype(int)
        tp = np.sum((pred == 1) & (y == 1))
        fp = np.sum((pred == 1) & (y == 0))
        nb_model.append(tp / N - (fp / N) * w)
        nb_all.append(prev - (1 - prev) * w)
    nb_model = np.array(nb_model)
    nb_all   = np.array(nb_all)
    nb_none  = np.zeros_like(thresholds)

    plt.figure(figsize=(8, 6))
    plt.plot(thresholds, nb_model, label='CNN model', color='#4c72b0', linewidth=2)
    plt.plot(thresholds, nb_all,   label='Treat All',  color='gray', linestyle='--')
    plt.plot(thresholds, nb_none,  label='Treat None', color='black', linestyle=':')
    plt.ylim(min(-0.05, nb_model.min()), max(nb_model.max(), prev) + 0.05)
    plt.xlabel("Threshold probability $p_t$")
    plt.ylabel("Net Benefit")
    plt.title("Decision Curve Analysis (patient-level)")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    p = os.path.join(config['output_dir'], "decision_curve_analysis.png")
    plt.savefig(p, dpi=150)
    plt.show()
    plt.close()
    print(f"  [SAVED] {p}")

    # Thresholds in the clinically relevant range (0.10–0.60) where the model
    # does worse than the better of the two baseline strategies
    baseline_best = np.maximum(nb_all, nb_none)
    harmful = thresholds[nb_model < baseline_best - 1e-9]
    harmful = harmful[(harmful >= 0.1) & (harmful <= 0.6)]
    if len(harmful) > 0:
        flag(f"Model Net Benefit dips BELOW Treat-All/Treat-None at thresholds "
             f"{np.round(harmful[[0, -1]], 2).tolist()} (range). Using the CNN there "
             f"may HARM clinical decision-making.")
    else:
        print("  [OK] Model Net Benefit exceeds both baselines across the "
              "clinically relevant threshold range (0.10-0.60).")

## PART C: CONFIDENCE INTERVALS

In [ ]:
# =============================================================================
# PART C — CONFIDENCE INTERVALS
# =============================================================================
# Three complementary interval types are reported:
#
#   (a) PATIENT LEVEL — the 40 patients are the independent units (one
#       prediction each), so classical binomial intervals are valid:
#         * Clopper-Pearson (exact) and Wilson score for accuracy,
#           sensitivity, specificity, PPV and NPV;
#         * DeLong (logit-transformed) for the AUC-ROC;
#         * a percentile bootstrap over patients for every metric, including
#           F1 and the Brier score, which have no closed-form interval.
#
#   (b) WINDOW LEVEL — windows of the same patient are correlated (B4), so a
#       binomial interval over all windows is far too narrow. The quoted
#       interval is a patient-cluster bootstrap: whole patients are resampled
#       with replacement and all their windows are kept together. The naive
#       Wilson interval is reported only as a reference.
#
#   (c) CROSS-FOLD — mean ± t(1-alpha/2, k-1) · SD / sqrt(k) over the k
#       fold-wise estimates: the interval for the mean of the CV estimator.
#
# Resampling is stratified by the patient's true class, so both classes are
# present in every bootstrap replicate and the AUC is always defined.

CI_METRIC_KEYS = ['accuracy', 'sensitivity', 'specificity', 'ppv', 'npv',
                  'f1', 'auc_roc', 'brier']


def _z_quantile(alpha):
    """Two-sided standard normal quantile z_{1-alpha/2}."""
    return float(stats.norm.ppf(1.0 - alpha / 2.0))


def _t_quantile(alpha, df):
    """Two-sided Student-t quantile t_{1-alpha/2, df}."""
    return float(stats.t.ppf(1.0 - alpha / 2.0, df))


def _wilson_ci(k, n, alpha=0.05):
    """Wilson score interval for a binomial proportion k/n (no continuity correction)."""
    if n == 0:
        return (float('nan'), float('nan'))
    z = _z_quantile(alpha)
    p = k / n
    denom  = 1.0 + z * z / n
    centre = (p + z * z / (2 * n)) / denom
    half   = (z / denom) * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return (float(max(0.0, centre - half)), float(min(1.0, centre + half)))


def _clopper_pearson_ci(k, n, alpha=0.05):
    """Exact (Clopper-Pearson) interval for a binomial proportion k/n."""
    if n == 0:
        return (float('nan'), float('nan'))
    lo = 0.0 if k == 0 else float(stats.beta.ppf(alpha / 2.0, k, n - k + 1))
    hi = 1.0 if k == n else float(stats.beta.ppf(1.0 - alpha / 2.0, k + 1, n - k))
    return (lo, hi)


def _compute_midrank(x):
    """Mid-ranks with ties averaged (helper for the DeLong variance)."""
    J = np.argsort(x, kind='mergesort')
    Z = np.asarray(x, dtype=float)[J]
    N = len(x)
    T = np.zeros(N, dtype=float)
    i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    T2 = np.empty(N, dtype=float)
    T2[J] = T
    return T2


def _delong_auc_ci(y_true, y_prob, alpha=0.05):
    """
    DeLong (1988) confidence interval for a single AUC, on the logit scale.

    Assumes independent observations, so it is used at the patient level only.
    Returns (auc, lo, hi, se).
    """
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    pos, neg = y_prob[y_true == 1], y_prob[y_true == 0]
    m, n = len(pos), len(neg)
    if m == 0 or n == 0:
        return (float('nan'),) * 4

    tx = _compute_midrank(pos)
    ty = _compute_midrank(neg)
    tz = _compute_midrank(np.concatenate([pos, neg]))

    auc = float(tz[:m].sum() / (m * n) - (m + 1.0) / (2.0 * n))
    v01 = (tz[:m] - tx) / n                    # placement values, cases
    v10 = 1.0 - (tz[m:] - ty) / m              # placement values, controls
    s01 = float(np.var(v01, ddof=1)) if m > 1 else 0.0
    s10 = float(np.var(v10, ddof=1)) if n > 1 else 0.0
    var = s01 / m + s10 / n
    se  = float(np.sqrt(max(var, 0.0)))

    z = _z_quantile(alpha)
    if se == 0 or auc <= 0.0 or auc >= 1.0:   # degenerate case: linear scale
        return (auc, float(max(0.0, auc - z * se)), float(min(1.0, auc + z * se)), se)

    # The logit transform keeps the interval inside [0, 1] and behaves better
    # for small samples.
    lg    = np.log(auc / (1.0 - auc))
    se_lg = se / (auc * (1.0 - auc))
    lo    = 1.0 / (1.0 + np.exp(-(lg - z * se_lg)))
    hi    = 1.0 / (1.0 + np.exp(-(lg + z * se_lg)))
    return (auc, float(lo), float(hi), se)


def _point_metrics_ci(true, pred, prob=None) -> dict:
    """Scalar metrics (incl. PPV, NPV, Brier) and confusion counts used for the intervals."""
    true = np.asarray(true).astype(int)
    pred = np.asarray(pred).astype(int)
    tp = int(np.sum((pred == 1) & (true == 1)))
    tn = int(np.sum((pred == 0) & (true == 0)))
    fp = int(np.sum((pred == 1) & (true == 0)))
    fn = int(np.sum((pred == 0) & (true == 1)))
    out = {
        'accuracy':    (tp + tn) / max(tp + tn + fp + fn, 1),
        'sensitivity': tp / (tp + fn) if (tp + fn) > 0 else float('nan'),
        'specificity': tn / (tn + fp) if (tn + fp) > 0 else float('nan'),
        'ppv':         tp / (tp + fp) if (tp + fp) > 0 else float('nan'),
        'npv':         tn / (tn + fn) if (tn + fn) > 0 else float('nan'),
        'f1':          (2 * tp) / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else float('nan'),
    }
    if prob is not None:
        prob = np.asarray(prob, dtype=float)
        try:
            out['auc_roc'] = float(roc_auc_score(true, prob))
        except ValueError:
            out['auc_roc'] = float('nan')
        out['brier'] = float(np.mean((prob - true) ** 2))
    else:
        out['auc_roc'] = float('nan')
        out['brier']   = float('nan')
    out['_counts'] = {'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn}
    return out


def _cluster_bootstrap_ci(true, pred, prob, groups, config, label=""):
    """
    Percentile bootstrap that resamples patients (clusters).

    `groups` holds the patient ID of every row. With patient-level input each
    patient owns one row, so this is an ordinary bootstrap; with window-level
    input all windows of a resampled patient are carried over together, which
    accounts for the intra-patient correlation (B4).
    """
    true   = np.asarray(true).astype(int)
    pred   = np.asarray(pred).astype(int)
    prob   = np.asarray(prob, dtype=float)
    groups = np.asarray(groups).astype(str)

    alpha  = config['ci_alpha']
    n_boot = config['ci_n_boot']
    rng    = np.random.RandomState(config['ci_bootstrap_seed'])

    uniq       = np.unique(groups)
    idx_of     = {g: np.where(groups == g)[0] for g in uniq}
    grp_label  = {g: int(np.round(true[idx_of[g]].mean())) for g in uniq}   # one label per patient
    pos_groups = np.array([g for g in uniq if grp_label[g] == 1])
    neg_groups = np.array([g for g in uniq if grp_label[g] == 0])

    stratified = bool(config['ci_stratified_boot']) and len(pos_groups) > 0 and len(neg_groups) > 0
    samples    = defaultdict(list)

    for _ in range(n_boot):
        if stratified:
            gs = np.concatenate([
                rng.choice(pos_groups, size=len(pos_groups), replace=True),
                rng.choice(neg_groups, size=len(neg_groups), replace=True),
            ])
        else:
            gs = rng.choice(uniq, size=len(uniq), replace=True)
        idx = np.concatenate([idx_of[g] for g in gs])
        m   = _point_metrics_ci(true[idx], pred[idx], prob[idx])
        for k in CI_METRIC_KEYS:
            samples[k].append(m[k])

    point = _point_metrics_ci(true, pred, prob)
    lo_q, hi_q = 100 * alpha / 2.0, 100 * (1 - alpha / 2.0)

    res = {}
    for k in CI_METRIC_KEYS:
        arr = np.asarray(samples[k], dtype=float)
        arr = arr[np.isfinite(arr)]
        if arr.size == 0:
            res[k] = {'point': point[k], 'lo': float('nan'), 'hi': float('nan'),
                      'se_boot': float('nan'), 'n_valid_boot': 0}
            continue
        res[k] = {
            'point':        float(point[k]),
            'lo':           float(np.percentile(arr, lo_q)),
            'hi':           float(np.percentile(arr, hi_q)),
            'se_boot':      float(np.std(arr, ddof=1)) if arr.size > 1 else float('nan'),
            'n_valid_boot': int(arr.size),
        }
    res['_counts']  = point['_counts']
    res['_n_units'] = int(len(uniq))
    res['_n_rows']  = int(len(true))
    res['_label']   = label
    return res


def _binomial_cis_from_counts(counts, alpha):
    """Wilson and Clopper-Pearson intervals for the count-based proportions."""
    tp, tn, fp, fn = counts['tp'], counts['tn'], counts['fp'], counts['fn']
    defs = {
        'accuracy':    (tp + tn, tp + tn + fp + fn),
        'sensitivity': (tp,      tp + fn),
        'specificity': (tn,      tn + fp),
        'ppv':         (tp,      tp + fp),
        'npv':         (tn,      tn + fn),
    }
    out = {}
    for metric, (k, n) in defs.items():
        out[metric] = {
            'k': int(k), 'n': int(n),
            'wilson':          _wilson_ci(k, n, alpha),
            'clopper_pearson': _clopper_pearson_ci(k, n, alpha),
        }
    return out


def _fmt_ci(point, lo, hi, dec=3):
    """Format 'estimate [lo, hi]'."""
    if not np.isfinite(point):
        return "n/a"
    return f"{point:.{dec}f} [{lo:.{dec}f}, {hi:.{dec}f}]"


def _crossfold_mean_ci(values, alpha):
    """Mean ± t(1-alpha/2, k-1) · SD / sqrt(k) over the k fold-wise estimates."""
    v = np.asarray([x for x in values if np.isfinite(x)], dtype=float)
    k = v.size
    if k == 0:
        return dict(mean=float('nan'), sd=float('nan'), sem=float('nan'),
                    lo=float('nan'), hi=float('nan'), k=0)
    if k == 1:
        return dict(mean=float(v[0]), sd=float('nan'), sem=float('nan'),
                    lo=float('nan'), hi=float('nan'), k=1)
    mean = float(v.mean())
    sd   = float(v.std(ddof=1))
    sem  = sd / np.sqrt(k)
    t    = _t_quantile(alpha, k - 1)
    return dict(mean=mean, sd=sd, sem=float(sem),
                lo=float(mean - t * sem), hi=float(mean + t * sem), k=int(k))


def _plot_forest(entries, title, filename, config):
    """Horizontal forest plot: point estimate and confidence interval per metric."""
    entries = [e for e in entries if np.isfinite(e['point'])]
    if not entries:
        return
    labels = [e['label'] for e in entries]
    pts    = np.array([e['point'] for e in entries], dtype=float)
    los    = np.array([e['lo']    for e in entries], dtype=float)
    his    = np.array([e['hi']    for e in entries], dtype=float)
    los    = np.where(np.isfinite(los), los, pts)
    his    = np.where(np.isfinite(his), his, pts)

    ypos = np.arange(len(entries))[::-1]
    fig, ax = plt.subplots(figsize=(9, 0.55 * len(entries) + 2.2))
    ax.errorbar(pts, ypos, xerr=[pts - los, his - pts], fmt='o',
                color='#4c72b0', ecolor='#4c72b0', elinewidth=2,
                capsize=4, markersize=7)
    for x, y in zip(pts, ypos):
        ax.text(x, y + 0.18, f"{x:.3f}", ha='center', va='bottom', fontsize=8)
    ax.axvline(0.5, color='gray', linestyle='--', alpha=0.6, label='Chance (0.50)')
    ax.set_yticks(ypos)
    ax.set_yticklabels(labels, fontsize=9)
    ax.set_xlim(0, 1.02)
    ax.set_xlabel(f"Estimate with {int((1 - config['ci_alpha']) * 100)}% confidence interval")
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    ax.legend(fontsize=8, loc='lower left')
    plt.tight_layout()
    p = os.path.join(config['output_dir'], filename)
    plt.savefig(p, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"  [SAVED] {p}")


def _dca_bootstrap_band(agg, config):
    """Patient-bootstrap confidence band around the net-benefit curve (B7)."""
    y    = np.asarray(agg['true']).astype(int)
    prob = np.asarray(agg['mean_prob'], dtype=float)
    thresholds = np.array(config['dca_thresholds'], dtype=float)
    alpha  = config['ci_alpha']
    n_boot = config['ci_n_boot']
    rng    = np.random.RandomState(config['ci_bootstrap_seed'] + 1)

    def _nb(yy, pp):
        N = len(yy)
        w = thresholds / (1.0 - thresholds)
        pred = (pp[None, :] >= thresholds[:, None]).astype(int)
        tp = (pred & (yy[None, :] == 1)).sum(axis=1)
        fp = (pred & (yy[None, :] == 0)).sum(axis=1)
        return tp / N - (fp / N) * w

    nb_point = _nb(y, prob)
    pos_idx, neg_idx = np.where(y == 1)[0], np.where(y == 0)[0]
    draws = np.empty((n_boot, len(thresholds)), dtype=float)
    for b in range(n_boot):
        idx = np.concatenate([
            rng.choice(pos_idx, size=len(pos_idx), replace=True),
            rng.choice(neg_idx, size=len(neg_idx), replace=True),
        ])
        draws[b] = _nb(y[idx], prob[idx])
    lo = np.percentile(draws, 100 * alpha / 2.0,       axis=0)
    hi = np.percentile(draws, 100 * (1 - alpha / 2.0), axis=0)

    prev   = y.mean()
    nb_all = prev - (1 - prev) * (thresholds / (1 - thresholds))

    plt.figure(figsize=(8, 6))
    plt.plot(thresholds, nb_point, label='CNN model', color='#4c72b0', linewidth=2)
    plt.fill_between(thresholds, lo, hi, color='#4c72b0', alpha=0.20,
                     label=f"{int((1-alpha)*100)}% bootstrap CI")
    plt.plot(thresholds, nb_all, label='Treat All', color='gray', linestyle='--')
    plt.plot(thresholds, np.zeros_like(thresholds), label='Treat None',
             color='black', linestyle=':')
    plt.ylim(min(-0.05, float(np.min(lo))), max(float(np.max(hi)), prev) + 0.05)
    plt.xlabel("Threshold probability $p_t$")
    plt.ylabel("Net Benefit")
    plt.title(f"Decision Curve Analysis with {int((1-alpha)*100)}% CI (patient-level)")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    p = os.path.join(config['output_dir'], "decision_curve_analysis_with_ci.png")
    plt.savefig(p, dpi=150)
    plt.show()
    plt.close()
    print(f"  [SAVED] {p}")

    return pd.DataFrame({'threshold': thresholds, 'net_benefit': nb_point,
                         'nb_ci_lo': lo, 'nb_ci_hi': hi, 'nb_treat_all': nb_all})


def compute_confidence_intervals(fold_results, pooled, agg, per_fold_df, config):
    """
    Part C: confidence intervals for every headline metric.

    Parameters
    ----------
    fold_results : list       output of run_inference_all_folds()
    pooled       : dict       pooled window-level arrays from evaluate_performance()
    agg          : dict       pooled patient-level arrays from evaluate_performance()
    per_fold_df  : DataFrame  per-fold metrics from analyze_crossfold_stability()
                              (None = rebuild from fold_results)

    Returns
    -------
    ci_df : tidy table of all intervals (also saved as confidence_intervals.csv)
    store : nested dict of all intervals (also saved as confidence_intervals.json)
    """
    conf_pct = int((1 - config['ci_alpha']) * 100)
    banner(f"C  CONFIDENCE INTERVALS ({conf_pct}%: binomial, DeLong, "
           f"patient-cluster bootstrap)", "#")
    print(f"  Bootstrap replicates : {config['ci_n_boot']} "
          f"(seed {config['ci_bootstrap_seed']}, "
          f"{'class-stratified' if config['ci_stratified_boot'] else 'unstratified'} "
          f"patient resampling)")

    alpha = config['ci_alpha']
    rows  = []   # -> confidence_intervals.csv
    store = {}   # -> confidence_intervals.json

    def _add_row(level, aggregation, metric, method, point, lo, hi, n_units, extra=""):
        rows.append({
            'level': level, 'aggregation': aggregation, 'metric': metric,
            'method': method, 'estimate': point, 'ci_lo': lo, 'ci_hi': hi,
            'ci_width': (hi - lo) if (np.isfinite(hi) and np.isfinite(lo)) else float('nan'),
            'n_units': n_units, 'confidence': 1 - alpha, 'note': extra,
        })

    # ── (a) Patient level ──────────────────────────────────────────────────
    pat_true = np.asarray(agg['true']).astype(int)
    pat_prob = np.asarray(agg['mean_prob'], dtype=float)
    pat_ids  = np.asarray(agg['pat_order']).astype(str)

    for agg_name, pat_pred in [('mean_prob', np.asarray(agg['mean_pred']).astype(int)),
                               ('majority',  np.asarray(agg['maj_pred']).astype(int))]:
        boot = _cluster_bootstrap_ci(pat_true, pat_pred, pat_prob, pat_ids, config,
                                     label=f"patient/{agg_name}")
        binom = _binomial_cis_from_counts(boot['_counts'], alpha)
        auc, auc_lo, auc_hi, auc_se = _delong_auc_ci(pat_true, pat_prob, alpha)

        print(f"\n  -- PATIENT LEVEL - {agg_name} aggregation "
              f"(n = {boot['_n_units']} patients) --")
        print(f"     {'metric':<12} {'estimate [ ' + str(conf_pct) + '% CI ] (bootstrap)':<30}"
              f" {'exact / analytic interval':<30}")
        for metric in ['accuracy', 'sensitivity', 'specificity', 'ppv', 'npv']:
            b  = boot[metric]
            cp = binom[metric]['clopper_pearson']
            wl = binom[metric]['wilson']
            print(f"     {metric:<12} {_fmt_ci(b['point'], b['lo'], b['hi']):<30} "
                  f"CP {_fmt_ci(b['point'], cp[0], cp[1]):<27} "
                  f"(k={binom[metric]['k']}/{binom[metric]['n']})")
            _add_row('patient', agg_name, metric, 'bootstrap_percentile_patient',
                     b['point'], b['lo'], b['hi'], boot['_n_units'])
            _add_row('patient', agg_name, metric, 'clopper_pearson_exact',
                     b['point'], cp[0], cp[1], binom[metric]['n'])
            _add_row('patient', agg_name, metric, 'wilson_score',
                     b['point'], wl[0], wl[1], binom[metric]['n'])

        for metric in ['f1', 'brier']:
            b = boot[metric]
            print(f"     {metric:<12} {_fmt_ci(b['point'], b['lo'], b['hi']):<30} "
                  f"(bootstrap only - no closed form)")
            _add_row('patient', agg_name, metric, 'bootstrap_percentile_patient',
                     b['point'], b['lo'], b['hi'], boot['_n_units'])

        b = boot['auc_roc']
        print(f"     {'auc_roc':<12} {_fmt_ci(b['point'], b['lo'], b['hi']):<30} "
              f"DeLong {_fmt_ci(auc, auc_lo, auc_hi):<23} (SE={auc_se:.4f})")
        _add_row('patient', agg_name, 'auc_roc', 'bootstrap_percentile_patient',
                 b['point'], b['lo'], b['hi'], boot['_n_units'])
        _add_row('patient', agg_name, 'auc_roc', 'delong_logit',
                 auc, auc_lo, auc_hi, boot['_n_units'], f"SE={auc_se:.4f}")

        store[f'patient_{agg_name}'] = {
            'n_patients': boot['_n_units'],
            'counts':     boot['_counts'],
            'bootstrap':  {k: boot[k] for k in CI_METRIC_KEYS},
            'binomial':   {k: {'k': v['k'], 'n': v['n'],
                               'wilson': list(v['wilson']),
                               'clopper_pearson': list(v['clopper_pearson'])}
                           for k, v in binom.items()},
            'delong_auc': {'auc': auc, 'lo': auc_lo, 'hi': auc_hi, 'se': auc_se},
        }

    # Forest plot of the headline (mean-probability) patient-level results
    pm = store['patient_mean_prob']['bootstrap']
    _plot_forest(
        [{'label': f"{k.replace('_', ' ').upper()} (patient, mean-prob)",
          'point': pm[k]['point'], 'lo': pm[k]['lo'], 'hi': pm[k]['hi']}
         for k in ['accuracy', 'sensitivity', 'specificity', 'ppv', 'npv', 'f1', 'auc_roc']],
        f"Patient-Level Performance with {conf_pct}% Bootstrap CIs "
        f"(n={store['patient_mean_prob']['n_patients']} patients)",
        "confidence_intervals_forest_patient.png", config)

    # ── (b) Window level: patient-cluster bootstrap vs. naive binomial ─────
    w_true = np.asarray(pooled['labels']).astype(int)
    w_pred = np.asarray(pooled['preds']).astype(int)
    w_prob = np.asarray(pooled['probs'], dtype=float)
    w_pat  = np.asarray(pooled['pat_ids']).astype(str)

    wboot  = _cluster_bootstrap_ci(w_true, w_pred, w_prob, w_pat, config, label="window")
    wbinom = _binomial_cis_from_counts(wboot['_counts'], alpha)

    print(f"\n  -- WINDOW LEVEL (n = {wboot['_n_rows']} windows from "
          f"{wboot['_n_units']} patients) --")
    print("     Naive binomial intervals assume i.i.d. windows and are therefore")
    print("     ANTI-CONSERVATIVE here; quote the cluster-bootstrap intervals.")
    for metric in ['accuracy', 'sensitivity', 'specificity', 'ppv', 'npv']:
        b  = wboot[metric]
        wl = wbinom[metric]['wilson']
        naive_w   = wl[1] - wl[0]
        cluster_w = b['hi'] - b['lo']
        infl = (cluster_w / naive_w) if naive_w > 0 else float('nan')
        print(f"     {metric:<12} cluster {_fmt_ci(b['point'], b['lo'], b['hi']):<26} "
              f"| naive Wilson [{wl[0]:.3f}, {wl[1]:.3f}] "
              f"| width x{infl:.1f}")
        _add_row('window', 'none', metric, 'cluster_bootstrap_patient',
                 b['point'], b['lo'], b['hi'], wboot['_n_units'],
                 f"width inflation vs naive Wilson: x{infl:.2f}")
        _add_row('window', 'none', metric, 'naive_wilson_iid_reference',
                 b['point'], wl[0], wl[1], wbinom[metric]['n'],
                 "NOT valid under intra-patient correlation")

    for metric in ['f1', 'auc_roc', 'brier']:
        b = wboot[metric]
        print(f"     {metric:<12} cluster {_fmt_ci(b['point'], b['lo'], b['hi'])}")
        _add_row('window', 'none', metric, 'cluster_bootstrap_patient',
                 b['point'], b['lo'], b['hi'], wboot['_n_units'])

    store['window'] = {
        'n_windows':  wboot['_n_rows'],
        'n_patients': wboot['_n_units'],
        'counts':     wboot['_counts'],
        'cluster_bootstrap': {k: wboot[k] for k in CI_METRIC_KEYS},
        'naive_binomial_reference': {
            k: {'k': v['k'], 'n': v['n'], 'wilson': list(v['wilson']),
                'clopper_pearson': list(v['clopper_pearson'])}
            for k, v in wbinom.items()},
    }

    # ── (c) Cross-fold: t-interval for the mean of the fold-wise estimates ─
    if per_fold_df is None:
        per_fold_df = build_per_fold_table(fold_results)

    print(f"\n  -- CROSS-FOLD mean +/- t({conf_pct}%, df=k-1) * SD/sqrt(k) --")
    store['crossfold'] = {}
    for level in per_fold_df['level'].unique():
        sub = per_fold_df[per_fold_df['level'] == level]
        store['crossfold'][level] = {}
        for metric in ['accuracy', 'sensitivity', 'specificity', 'f1', 'auc_roc']:
            st = _crossfold_mean_ci(sub[metric].values, alpha)
            print(f"     {level:<18} {metric:<12} "
                  f"{st['mean']:.4f} +/- {st['sd']:.4f} (SD)  ->  "
                  f"CI [{st['lo']:.4f}, {st['hi']:.4f}]  (k={st['k']} folds)")
            _add_row(level, 'per_fold_mean', metric, 't_interval_crossfold',
                     st['mean'], st['lo'], st['hi'], st['k'],
                     f"SD={st['sd']:.4f}, SEM={st['sem']:.4f}")
            store['crossfold'][level][metric] = st

    # ── (d) Decision curve: bootstrap confidence band ──────────────────────
    print(f"\n  -- DECISION CURVE - {conf_pct}% bootstrap band --")
    dca_df = _dca_bootstrap_band(agg, config)
    dca_path = os.path.join(config['output_dir'], "decision_curve_analysis_ci.csv")
    dca_df.to_csv(dca_path, index=False)
    print(f"  [SAVED] {dca_path}")

    # ── (e) Save, and flag imprecise estimates ─────────────────────────────
    ci_df = pd.DataFrame(rows)
    csv_path = os.path.join(config['output_dir'], "confidence_intervals.csv")
    ci_df.to_csv(csv_path, index=False)
    print(f"\n  [SAVED] {csv_path}")

    json_path = os.path.join(config['output_dir'], "confidence_intervals.json")
    with open(json_path, 'w') as f:
        json.dump({
            'confidence_level':   1 - alpha,
            'n_bootstrap':        config['ci_n_boot'],
            'bootstrap_seed':     config['ci_bootstrap_seed'],
            'bootstrap_unit':     'patient (cluster resampling)',
            'stratified':         bool(config['ci_stratified_boot']),
            'methods': {
                'proportions_patient_level': 'Clopper-Pearson exact + Wilson score',
                'auc_patient_level':         'DeLong (logit-transformed) + bootstrap',
                'window_level':              'patient-cluster bootstrap (naive Wilson '
                                             'reported only as an i.i.d. reference)',
                'crossfold':                 'Student-t interval on the mean of the '
                                             'k fold-wise estimates',
            },
            'results': store,
        }, f, indent=2, default=float)
    print(f"  [SAVED] {json_path}")

    # Precision check on the headline (patient-level, mean-probability) intervals
    wide = []
    for metric in ['accuracy', 'sensitivity', 'specificity', 'auc_roc']:
        b = store['patient_mean_prob']['bootstrap'][metric]
        if np.isfinite(b['lo']) and np.isfinite(b['hi']):
            if (b['hi'] - b['lo']) > config['ci_width_warn']:
                wide.append(f"{metric} [{b['lo']:.3f}, {b['hi']:.3f}] "
                            f"(width {b['hi'] - b['lo']:.3f})")
    if wide:
        flag(f"Wide patient-level {conf_pct}% CIs (> {config['ci_width_warn']}): "
             f"{'; '.join(wide)}. With only {store['patient_mean_prob']['n_patients']} "
             f"patients the point estimates are IMPRECISE - report intervals, not "
             f"point values alone, and avoid claiming superiority over other models.")
    else:
        print(f"  [OK] All headline patient-level CIs are narrower than "
              f"{config['ci_width_warn']}.")

    # Does the AUC interval exclude chance?
    auc_lo = store['patient_mean_prob']['delong_auc']['lo']
    if np.isfinite(auc_lo) and auc_lo <= 0.5:
        flag(f"Patient-level AUC {conf_pct}% CI includes 0.50 "
             f"(lower bound {auc_lo:.3f}). Discrimination is NOT statistically "
             f"distinguishable from chance at the {conf_pct}% level.")
    else:
        print(f"  [OK] Patient-level AUC CI excludes chance (lower bound {auc_lo:.3f}).")

    return ci_df, store

## RUN — Load Inputs and Re-run Inference

In [ ]:
print("=" * 60)
print("Script 3: Evaluation")
print("Dilated ResNet + Temporal Attention  |  Robustness Diagnostics  |  CIs")
print("=" * 60)

start_time = time.perf_counter()
print(f"Start: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

meta         = load_cv_metadata(CONFIG)
dataset      = load_dataset(CONFIG)
fold_results = run_inference_all_folds(meta, dataset, CONFIG)

## RUN — Part A: Performance Evaluation

In [ ]:
pooled, agg = evaluate_performance(fold_results, meta, CONFIG)

## RUN — Part B: Robustness Diagnostics

In [ ]:
per_fold_df, _ = analyze_crossfold_stability(fold_results, CONFIG)        # B1
analyze_embeddings_tsne(fold_results, CONFIG)                             # B2
run_permutation_test(fold_results, CONFIG)                                # B3
analyze_intra_patient_correlation(dataset, CONFIG)                        # B4
analyze_aggregation_and_calibration(pooled, agg, CONFIG)                  # B5
analyze_integrated_gradients(fold_results, CONFIG)                        # B6
run_decision_curve_analysis(agg, CONFIG)                                  # B7

## RUN — Part C: Confidence Intervals

In [ ]:
compute_confidence_intervals(fold_results, pooled, agg, per_fold_df, CONFIG)

elapsed = time.perf_counter() - start_time
print(f"\n[DONE] All evaluation outputs saved to: {CONFIG['output_dir']}")
print(f"End: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Total time: {elapsed:.1f}s ({elapsed / 60:.1f} min)")
print("Review any [FLAG] blocks above for robustness warnings.")